# Pipeline walkthrough: what each modeling stage actually does

`src/train.py` prints a condensed version of this same walkthrough every time it runs (see its `print_stage_by_stage_walkthrough` function). This notebook is the same idea, in more detail, so you can pause at each cell, inspect the data, and change things to see what happens.

Run this **after** `python src/data_download.py` and `python src/preprocess.py`. It is not called by `train.py`, `run_pipeline.py`, or any other script -- it's a standalone teaching aid. For *why* each stage was chosen (not just what it does), see `../model_plan.md`.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.config import TRAIN_PARQUET, get_spark_session

spark = get_spark_session("pipeline-walkthrough")
df = spark.read.parquet(str(TRAIN_PARQUET))
sample = df.limit(5)
# review_text = title + body, combined by preprocess.py -- this is what the
# model actually tokenizes below, not "text" alone. See ../model_plan.md.
sample.select("label", "title", "text", "review_text").show(truncate=50)

## Stage 1 -- Tokenizer

Splits each review's raw text into a list of individual words. Models work with numbers, not sentences -- this is the first step that breaks free-text into discrete units we can eventually turn into numeric features.

In [ ]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="review_text", outputCol="words")
tokenized = tokenizer.transform(sample)
tokenized.select("review_text", "words").show(truncate=60)

## Stage 2 -- StopWordsRemover

Drops extremely common words ("the", "is", "a", ...) that carry almost no sentiment signal on their own. This shrinks the vocabulary and lets the model spend its capacity on words that actually distinguish positive from negative reviews.

In [3]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
filtered = remover.transform(tokenized)
filtered.select("words", "filtered_words").show(truncate=60)

before = filtered.selectExpr("size(words) as n").toPandas()["n"]
after = filtered.selectExpr("size(filtered_words) as n").toPandas()["n"]
print(f"Average word count before: {before.mean():.1f}, after removing stopwords: {after.mean():.1f}")

+------------------------------------------------------------+------------------------------------------------------------+
|                                                       words|                                              filtered_words|
+------------------------------------------------------------+------------------------------------------------------------+
|[this, sound, track, was, beautiful!, it, paints, the, se...|[sound, track, beautiful!, paints, senery, mind, well, re...|
|[i'm, reading, a, lot, of, reviews, saying, that, this, i...|[reading, lot, reviews, saying, best, 'game, soundtrack',...|
|[this, soundtrack, is, my, favorite, music, of, all, time...|[soundtrack, favorite, music, time,, hands, down., intens...|
|[i, truly, like, this, soundtrack, and, i, enjoy, video, ...|[truly, like, soundtrack, enjoy, video, game, music., pla...|
|[if, you've, played, the, game,, you, know, how, divine, ...|[played, game,, know, divine, music, is!, every, single, ...|
+-------

Average word count before: 97.4, after removing stopwords: 53.6


## Stage 3 -- HashingTF

Converts the remaining words into a fixed-size numeric vector by hashing each word to a bucket index. The alternative (`CountVectorizer`) builds an explicit word-to-column vocabulary, which needs a full pass over the data and grows without bound as new words appear. Hashing trades a small, usually negligible risk of two different words landing in the same bucket for a fixed memory footprint and no separate vocabulary-building pass -- a good trade at this dataset's scale.

A small `numFeatures` is used below only so the printed vectors stay readable -- the real pipeline in `train.py` uses much larger values (tuned via cross-validation, see the next section).

In [4]:
from pyspark.ml.feature import HashingTF

hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=2 ** 8)
hashed = hashing_tf.transform(filtered)
hashed.select("filtered_words", "raw_features").show(truncate=60)

+------------------------------------------------------------+------------------------------------------------------------+
|                                              filtered_words|                                                raw_features|
+------------------------------------------------------------+------------------------------------------------------------+
|[sound, track, beautiful!, paints, senery, mind, well, re...|(256,[1,5,7,17,23,24,26,28,34,38,45,53,70,84,106,116,118,...|
|[reading, lot, reviews, saying, best, 'game, soundtrack',...|(256,[0,3,4,25,31,43,47,48,55,63,65,66,73,74,89,95,97,113...|
|[soundtrack, favorite, music, time,, hands, down., intens...|(256,[0,2,6,8,9,14,16,18,24,28,30,31,32,39,41,42,43,44,47...|
|[truly, like, soundtrack, enjoy, video, game, music., pla...|(256,[14,28,42,44,47,49,52,56,63,65,69,76,81,82,87,95,97,...|
|[played, game,, know, divine, music, is!, every, single, ...|(256,[1,4,9,34,41,43,59,69,70,71,79,82,99,100,102,108,113...|
+-------

## Stage 4 -- IDF (Inverse Document Frequency)

Re-weights the hashed term-frequency vector so words appearing in *almost every* review count for less, and rarer-but-informative words count for more. TF alone treats "good" and "phenomenal" as equally important if they appear the same number of times; IDF recognizes that rarer words are often more discriminative of sentiment.

In [5]:
from pyspark.ml.feature import IDF

idf_model = IDF(inputCol="raw_features", outputCol="features").fit(hashed)
reweighted = idf_model.transform(hashed)
reweighted.select("raw_features", "features").show(truncate=60)

+------------------------------------------------------------+------------------------------------------------------------+
|                                                raw_features|                                                    features|
+------------------------------------------------------------+------------------------------------------------------------+
|(256,[1,5,7,17,23,24,26,28,34,38,45,53,70,84,106,116,118,...|(256,[1,5,7,17,23,24,26,28,34,38,45,53,70,84,106,116,118,...|
|(256,[0,3,4,25,31,43,47,48,55,63,65,66,73,74,89,95,97,113...|(256,[0,3,4,25,31,43,47,48,55,63,65,66,73,74,89,95,97,113...|
|(256,[0,2,6,8,9,14,16,18,24,28,30,31,32,39,41,42,43,44,47...|(256,[0,2,6,8,9,14,16,18,24,28,30,31,32,39,41,42,43,44,47...|
|(256,[14,28,42,44,47,49,52,56,63,65,69,76,81,82,87,95,97,...|(256,[14,28,42,44,47,49,52,56,63,65,69,76,81,82,87,95,97,...|
|(256,[1,4,9,34,41,43,59,69,70,71,79,82,99,100,102,108,113...|(256,[1,4,9,34,41,43,59,69,70,71,79,82,99,100,102,108,113...|
+-------

## Stage 5 -- LogisticRegression

The classifier itself. Chosen over more complex models as a fast, interpretable, strong baseline for text classification: training is quick even on millions of sparse rows, and each feature gets a signed weight you can inspect. Below, we fit it directly on this 5-row toy sample just to see the mechanics -- `train.py`'s real fit uses `CrossValidator` across the full training split with a hyperparameter grid, not a hand-picked configuration like this.

In [6]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
toy_model = lr.fit(reweighted)
toy_model.transform(reweighted).select("label", "prediction", "probability").show(truncate=60)

26/07/27 09:01:22 WARN Instrumentation: [3b2a6716] All labels are the same value and fitIntercept=true, so the coefficients will be zeros. Training is not needed.


+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
+-----+----------+-----------+



## Putting it together, and what `train.py` does differently

The real training run wraps all five stages above in a single `pyspark.ml.Pipeline`, then wraps *that* in a `CrossValidator` that:

- tries a grid of `HashingTF.numFeatures` / `LogisticRegression.regParam` / `elasticNetParam` combinations,
- evaluates each combination with 5-fold cross-validation on an 80% training split,
- refits the best combination on the full 80% split,
- checks that winner once against a held-out 20% validation split,
- and logs everything (including an untuned baseline for comparison) to MLflow.

See `../model_plan.md` for why each of those choices was made, and run `python src/train.py` to see it end to end.

In [7]:
spark.stop()